# 09: Labeling the full corpus

**DATA 5500, Senior Design**

**Sara Choque Molina**

Notebook 08 settled the model: TF-IDF with logistic regression, a Cohen's kappa of 0.676 against the human labels. Here that model is trained on all 136 labeled articles and used to label the rest of the 2021 corpus, so the trend questions can be counted on classified articles instead of raw keyword matches.

A predicted label is not a verified one. The later blocks draw a stratified random sample of 50 predicted articles, read blind, and report the agreement between those readings and the model.

| File | Contents |
|---|---|
| `training_set.csv` | The 136 labeled articles with text, from notebook 07 |
| `articles_2021_clean.csv` | The 1,554 clean 2021 articles, from notebook 03 |
| `labels_predicted_full.csv` | Written here: every article in the corpus with a label, its source and the model's confidence |
| `09_verification_sample.xlsx` | Written here: the blind sample to read by hand |
| `09_verification_sample_key.csv` | Written here: the labels withheld from that sample |

## Block 1: The corpus, and what has already been read

The corpus and the labeled set are read together, and the articles a person has already labeled are set aside so the model is never asked to predict one of them.

Two things are checked before anything is predicted. The corpus has to split cleanly into the articles already read and the remainder, with the two counts adding back to the total. And no article in the remainder may repeat the text of a labeled one, since an identical copy would make a prediction look like a judgment when it is really a lookup.

In [1]:
import os

import numpy as np
import pandas as pd

FOLDER = r"C:\Users\sara_\Desktop\College\DATA 5500-01 Senior Des\data bases"

FIGURES = os.path.join(FOLDER, "figures")
os.makedirs(FIGURES, exist_ok=True)

LABELS = ["targeted_threat", "nontargeted_threat", "unrelated"]

training = pd.read_csv(os.path.join(FOLDER, "training_set.csv"))
corpus = pd.read_csv(os.path.join(FOLDER, "articles_2021_clean.csv"))[["url", "text"]].dropna()

rest = corpus[~corpus["url"].isin(training["url"])].copy()
found = corpus["url"].isin(training["url"]).sum()

print("Corpus articles with text:", len(corpus))
print("Labeled articles found in the corpus:", found, "of", len(training))
print("Left for the model:", len(rest))
print("Counts add back to the total:", found + len(rest) == len(corpus))
print()

labeled_texts = set(training["text"].str.strip())
print("Unlabeled articles repeating a labeled text:", rest["text"].str.strip().isin(labeled_texts).sum())
print("Duplicate texts inside the unlabeled group:", len(rest) - rest["text"].str.strip().nunique())
print("Duplicate URLs inside the unlabeled group:", len(rest) - rest["url"].nunique())

Corpus articles with text: 1554
Labeled articles found in the corpus: 136 of 136
Left for the model: 1418
Counts add back to the total: True

Unlabeled articles repeating a labeled text: 0
Duplicate texts inside the unlabeled group: 0
Duplicate URLs inside the unlabeled group: 0


**Results**

- The corpus holds 1,554 articles with usable text. All 136 labeled articles are present in it and the remainder is 1,418, so the two counts add back to the total.
- No article in the remainder repeats the text of a labeled one, and the remainder holds no duplicate texts or URLs of its own. Every prediction in this notebook is made on an article the model has not seen, and no article is counted twice.

## Block 2: Label the remaining articles

We fit the model notebook 08 settled on, same settings and same class weights, on all 136 labeled articles instead of four fifths at a time. Every article ends up in one table with a `source` column saying who assigned its label.

A model trained on a set that is 48 percent `targeted_threat` tends to produce about that share again, so the resemblance between the two distributions below says nothing about the corpus.claim.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight

model = make_pipeline(
    TfidfVectorizer(stop_words="english", min_df=2),
    LogisticRegression(max_iter=1000),
)

weights = compute_sample_weight("balanced", training["label"])
model.fit(training["text"], training["label"], logisticregression__sample_weight=weights)

rest["label"] = model.predict(rest["text"])
rest["source"] = "model"

human = training[["url", "label"]].copy()
human["source"] = "human"

full = pd.concat([human, rest[["url", "label", "source"]]], ignore_index=True)
full.to_csv(os.path.join(FOLDER, "labels_predicted_full.csv"), index=False)

print("Labeled by a person:", (full["source"] == "human").sum())
print("Labeled by the model:", (full["source"] == "model").sum())
print("Total:", len(full))
print()
print("Saved: labels_predicted_full.csv")

pd.DataFrame({
    "articles": full["label"].value_counts().reindex(LABELS),
    "% of corpus": (full["label"].value_counts(normalize=True).reindex(LABELS) * 100).round(1),
    "% of the 136 read by a person": (
        training["label"].value_counts(normalize=True).reindex(LABELS) * 100).round(1),
})

Labeled by a person: 136
Labeled by the model: 1418
Total: 1554

Saved: labels_predicted_full.csv


,articles,% of corpus,% of the 136 read by a person
label,,,
targeted_threat,783,50.4,47.8
nontargeted_threat,417,26.8,25.0
unrelated,354,22.8,27.2


**Results**

- The predicted shares sit within two and a half points of the training distribution on the two threat classes, and four and a half points on `unrelated`. That is what a model trained on this set is expected to produce, so it is not a finding about the corpus yet. The verification sample decides whether it becomes one.

## Block 3: The verification sample

We draw 50 of the articles the model labeled, stratified by predicted label.

Eight of the fifty appear twice, under different row numbers and scattered by the shuffle. Agreement between those two readings measures how consistently we apply the codebook, separately from whether the model agrees with us.gless.

In [21]:
SAMPLE_SIZE = 50
REPEATS = 8
SEED = 124

sizes = (rest["label"].value_counts() / len(rest) * SAMPLE_SIZE).round().astype(int)

fresh = pd.concat([
    rest[rest["label"] == label].sample(n=sizes[label], random_state=SEED)
    for label in sorted(LABELS)
])
repeated = fresh.sample(n=REPEATS, random_state=SEED)

sheet = pd.concat([fresh, repeated]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sheet.insert(0, "id", range(1, len(sheet) + 1))

sheet[["id", "url", "label"]].to_csv(
    os.path.join(FOLDER, "09_verification_sample_key.csv"), index=False)

blind = sheet[["id", "url", "text"]].copy()
# Excel refuses a cell over 32,767 characters, so the longest articles are trimmed.
blind["text"] = blind["text"].str.slice(0, 30000)
blind["my_label"] = ""
blind["notes"] = ""
blind = blind[["id", "my_label", "notes", "url", "text"]]

path = os.path.join(FOLDER, "09_verification_sample.xlsx")

# A sheet already on disk may hold readings, so it is not rewritten.
if os.path.exists(path):
    print("Already on disk, left untouched: 09_verification_sample.xlsx")
else:
    blind.to_excel(path, index=False)
    print("Saved: 09_verification_sample.xlsx")

print()
print("Rows in the sheet:", len(sheet), "=", len(fresh), "articles plus", REPEATS, "repeats")
print(fresh["label"].value_counts().reindex(LABELS).to_string())

Already on disk, left untouched: 09_verification_sample.xlsx

Rows in the sheet: 58 = 50 articles plus 8 repeats
label
targeted_threat       25
nontargeted_threat    14
unrelated             11


**Results**

- The 50 match the composition of the predicted set to within a point and a half on every class.
- At 50 articles the agreement with the model carries a margin of 13.6% in the worst case, and 9.7% if it lands near 85%.csv`.

## Block 4: Agreement with the model, and with ourselves

Rows read as `unusable` count toward the total but we do not score them, since the model knows three labels and cannot produce that one.

We take two measures. For agreement with the model we use the first reading of each article. For consistency we use the eight repeated articles, where both copies carry a reading.

In [23]:
from sklearn.metrics import accuracy_score, cohen_kappa_score

READABLE = LABELS + ["unusable"]

filled = pd.read_excel(os.path.join(FOLDER, "09_verification_sample.xlsx"))
key = pd.read_csv(os.path.join(FOLDER, "09_verification_sample_key.csv"))

filled["my_label"] = filled["my_label"].astype(str).str.strip().str.lower()
read = filled.loc[filled["my_label"].isin(READABLE), ["id", "my_label"]].merge(
    key[["id", "url", "label"]], on="id")

# A typo would shrink the sample without showing it, so the count is always printed.
typos = filled[~filled["my_label"].isin(READABLE) & (filled["my_label"] != "nan")]

print("Rows read:", len(read), "of", len(filled))
print("Read as unusable, not scored:", int((read["my_label"] == "unusable").sum()))
print("Entries matching none of the four labels:", len(typos))

# The repeated articles, counted only where both copies carry a reading.
pairs = read[read.duplicated("url", keep=False)].groupby("url")["my_label"].agg(["nunique", "count"])
pairs = pairs[pairs["count"] == 2]

print()
print("Repeated articles read twice:", len(pairs))
print("Labeled the same way both times:", int((pairs["nunique"] == 1).sum()))

scored = read.drop_duplicates("url", keep="first")
scored = scored[scored["my_label"].isin(LABELS)]

print()
print("Scored against the model:", len(scored))
print("accuracy:", round(accuracy_score(scored["label"], scored["my_label"]), 3))
print("Cohen kappa:", round(cohen_kappa_score(scored["label"], scored["my_label"]), 3))
print()
print(pd.crosstab(scored["my_label"], scored["label"],
                  rownames=["read by hand"], colnames=["model"]).to_string())

# Collapsed to the one decision the trend questions depend on
def threat_or_not(column):
    return column.where(column == "unrelated", "threat")

coarse_mine = threat_or_not(scored["my_label"])
coarse_model = threat_or_not(scored["label"])

print()
print("Threat against not a threat")
print("accuracy:", round(accuracy_score(coarse_model, coarse_mine), 3))
print("Cohen kappa:", round(cohen_kappa_score(coarse_model, coarse_mine), 3))
print()
print(pd.crosstab(coarse_mine, coarse_model, rownames=["read by hand"], colnames=["model"]).to_string())

Rows read: 58 of 58
Read as unusable, not scored: 5
Entries matching none of the four labels: 0

Repeated articles read twice: 8
Labeled the same way both times: 6

Scored against the model: 45
accuracy: 0.756
Cohen kappa: 0.585

model               nontargeted_threat  targeted_threat  unrelated
read by hand                                                      
nontargeted_threat                  12                6          0
targeted_threat                      2               18          2
unrelated                            0                1          4

Threat against not a threat
accuracy: 0.933
Cohen kappa: 0.69

model         threat  unrelated
read by hand                   
threat            38          2
unrelated          1          4


**Results**

- 5 of the 58 rows are not news stories, 8.6%, which puts roughly 120 of the 1,418 predicted articles outside the corpus.
- The three label kappa is 0.091 below the 0.676 the model reached in cross validation on the labeled set. That is about the size of the spread the model shows between random splits, so the drop is real but modest.
- Collapsed to whether a threat was reported, the two differ on 3 of 45 articles. That is the decision the trend questions depend on.
- 11 of the 45 disagree across the three labels, and 8 of those sit on the line between `targeted_threat` and `nontargeted_threat`. Of the 8 repeated articles we labeled 6 the same way twice, and both that changed sit on that same line.

## Block 5: Whether confidence predicts where the model is right

The classifier reports a probability for the label it picks. We take that probability for all 1,418 predictions, then ask whether the 45 articles we read carry higher confidence where the model got them right.

If they do, a threshold lets the trend questions drop the predictions the model is least sure of.ame.

In [24]:
confidence = pd.DataFrame({
    "url": rest["url"].values,
    "confidence": model.predict_proba(rest["text"]).max(axis=1),
})

print("Confidence across the", len(confidence), "predictions")
print("median:", round(confidence["confidence"].median(), 3))
print("below 0.5:", int((confidence["confidence"] < 0.5).sum()), "of", len(confidence))
print()

checked = scored.merge(confidence, on="url")
checked["agreed"] = checked["my_label"] == checked["label"]
checked["agreed_coarse"] = threat_or_not(checked["my_label"]) == threat_or_not(checked["label"])

bands = pd.cut(checked["confidence"], [0, 0.5, 0.6, 1.0],
               labels=["below 0.5", "0.5 to 0.6", "above 0.6"])

table = checked.groupby(bands, observed=True).agg(
    articles=("agreed", "size"),
    three_labels=("agreed", "mean"),
    threat_or_not=("agreed_coarse", "mean"),
)
table["three_labels"] = (table["three_labels"] * 100).round(1)
table["threat_or_not"] = (table["threat_or_not"] * 100).round(1)

print("Share we agreed with, in percent, by the model's confidence")
print(table.to_string())

Confidence across the 1418 predictions
median: 0.508
below 0.5: 664 of 1418

Share we agreed with, in percent, by the model's confidence
            articles  three_labels  threat_or_not
confidence                                       
below 0.5         25          64.0           88.0
0.5 to 0.6         9          88.9          100.0
above 0.6         11          90.9          100.0


**Results**

- Confidence tracks correctness on both readings of the label, rising in the same direction across all three bands.
- Above 0.5 the model and the reading never disagreed on whether a threat was reported, across 20 articles. With that few, the figure says the error rate is low, not that it is zero.
- The weaker band is not a small corner: 664 of the 1,418 predictions sit below 0.5, and that is where the error concentrates.
- The trend questions can be counted twice, once on all predictions and once on the 754 at 0.5 or above, and the two curves compared.

## Block 6: The label file the trend questions read

We write one table for the whole corpus: url, label, whether a person or the model assigned it, and the model's confidence where there is one. Notebook 10 reads this file to count classified articles per day.alf.

In [26]:
final = full.merge(confidence, on="url", how="left")
final.to_csv(os.path.join(FOLDER, "labels_predicted_full.csv"), index=False)

print("Rows:", len(final))
print("With a confidence value:", int(final["confidence"].notna().sum()))
print("At 0.5 or above:", int((final["confidence"] >= 0.5).sum()))
print()
print("Saved: labels_predicted_full.csv")
final.sample(5, random_state=1)

Rows: 1554
With a confidence value: 1418
At 0.5 or above: 754

Saved: labels_predicted_full.csv


,url,label,source,confidence
995,https://patch.com/california/los-angeles/tikto...,nontargeted_threat,model,0.739129
396,https://lancasteronline.com/news/local/12-year...,targeted_threat,model,0.661113
1073,https://www.the-leader.com/story/opinion/colum...,nontargeted_threat,model,0.460594
647,https://www.inforum.com/news/education/7315739...,targeted_threat,model,0.644115
491,https://www.fox17online.com/rebound/state-of-e...,targeted_threat,model,0.370222


## Conclusions

| | accuracy | kappa |
|---|---|---|
| Three labels | 0.756 | 0.585 |
| Threat or not | 0.933 | 0.690 |

**We accept the predicted labels for the threat decision and not for the three way split.** Questions 4 and 5 count articles about threats per day, and that is the decision that verified. The share of the corpus falling into each of the three labels did not, so we do not report it. Those questions will be counted twice, on all predictions and on the stronger half, since confidence turned out to track where the model is right.

Four measurements point at the same place. The annotators split on `targeted_threat` against `nontargeted_threat` in round 1, the disputed articles cluster in that class, 8 of our 11 disagreements with the model sit on that line, and both repeated articles we labeled differently sit there too. The distinction is unstable within one reader as well as between reader and model.

**Limits**

- 50 articles and one reader. The worst case margin is 13.6%.
- 8.6% of the sample is not a news story. We know the rate, not which articles carry it.
- The reader adjudicated part of the training labels, so our reading and the convention the model learned are not independent. That pushes agreement up.
- Every article comes from the 2021 case.

**Next steps (#TODO later)**

- #TODO: Recover the publication date, which `articles_2021_clean.csv` does not carry, by joining on url from the GDELT export.
- #TODO: Redo Questions 4 and 5 on classified counts, once on all predictions and once on those at 0.5 or above.
- #TODO: Use SHAP with a text masker on the model for Question 2.
- #TODO: Settle whether the `targeted_threat` against `nontargeted_threat` distinction stays.